In [76]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
#import scipy
from scipy import stats
import seaborn as sns
from functools import reduce
import pathlib
from csv import reader
%matplotlib inline
import warnings
import xlsxwriter
from sklearn.decomposition import PCA
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from scipy.stats import pearsonr
import scipy.stats as stat
import dash_bio
import plotly.express as px
warnings.filterwarnings('ignore')

In [77]:
input_path = pathlib.Path("/Users/nropek/Dropbox (Dropbox @RU)/TurboID manuscript/Mass-spectrometry datasets/Chait-Cohen TurboID in Vitro")
output_path = pathlib.Path("/Users/nropek/Dropbox (Dropbox @RU)/TurboID manuscript/Mass-spectrometry datasets/TurboID_analysis/notebook_analysis/06_chait_cohen_data")

In [78]:
#get master list
master_list = pd.read_csv("/Users/nropek/Dropbox (Dropbox @RU)/TurboID manuscript/Mass-spectrometry datasets/TurboID_analysis/mouse_lists/master_list/mouse_master_table_for_ROC_analysis.csv", index_col=[0])
TP_list = master_list[master_list["annotation"] == "TP"]
#TP_list = TP_list["uniprot_id"].tolist()
FP_list = master_list[master_list["annotation"] == "FP"]
#FP_list = FP_list["uniprot_id"].tolist()

print("Number of Uniport IDs in TP list:", len(TP_list)) #3436
print("Number of Uniport IDs in FP list:", len(FP_list)) #4353

Number of Uniport IDs in TP list: 3436
Number of Uniport IDs in FP list: 4353


In [79]:
#get list of keratins uniprot ids 
keratins = master_list[master_list.keratin == True]
keratins_list = keratins.uniprot_id.tolist()

In [80]:
file_path_list = list(input_path.glob('*.xlsx'))

In [81]:
folder_path = output_path / "01_clean_df"
if not os.path.exists(folder_path):
    os.mkdir(folder_path)


clean_dfs = []

for file_path in file_path_list: 
    df = pd.read_excel(file_path)
    columns_to_keep_ibaq = list(df.filter(regex="iBAQ C").columns) #LFQ intensity
    columns_to_keep_lfq = list(df.filter(regex="LFQ intensity").columns) #LFQ intensity
    
    df_subset = df[columns_to_keep_ibaq + columns_to_keep_lfq + ["Protein IDs"]]
    df_subset.columns = df_subset.columns.str.replace(" intensity ", "_")
    df_subset.columns = df_subset.columns.str.replace("iBAQ ", "iBAQ_")
    df_subset.set_index("Protein IDs", inplace=True)
    df_subset = df_subset.add_prefix(file_path.stem.split(" ", 1)[0]+"_")
    df_subset.reset_index(inplace=True)
    df_subset["uniprot_id"] = df_subset["Protein IDs"].str.split("|").str[1]
    clean_dfs.append(df_subset)
    df_subset.to_csv(folder_path / (file_path.stem + ".csv"))
    

In [82]:
clean_dfs[1]

,Protein IDs,SubQ_iBAQ_Cre+_1,SubQ_iBAQ_Cre+_2,SubQ_iBAQ_Cre+_3,SubQ_iBAQ_Cre+_4,SubQ_iBAQ_Cre+_5,SubQ_iBAQ_Cre+_6,SubQ_iBAQ_Cre-_1,SubQ_iBAQ_Cre-_2,SubQ_iBAQ_Cre-_3,...,SubQ_LFQ_Cre+_4,SubQ_LFQ_Cre+_5,SubQ_LFQ_Cre+_6,SubQ_LFQ_Cre-_1,SubQ_LFQ_Cre-_2,SubQ_LFQ_Cre-_3,SubQ_LFQ_Cre-_4,SubQ_LFQ_Cre-_5,SubQ_LFQ_Cre-_6,uniprot_id
0,sp|Q61414|K1C15_MOUSE;CON__A2A4G1,0.0,0.000000e+00,0.0,21734.0,0.0,0.000000e+00,0.0,0.0,0.0,...,213210,0,0,0,0,0,0,0,0,Q61414
1,sp|Q9TTE1|SPA31_BOVIN;sp|A2I7N1|SPA35_BOVIN;sp...,376800.0,2.368200e+05,0.0,0.0,277280.0,0.000000e+00,118690.0,0.0,189530.0,...,0,0,0,0,0,1503000,0,0,0,Q9TTE1
2,tr|F1MVK1|F1MVK1_BOVIN;CON__ENSEMBL:ENSBTAP000...,6570400.0,8.288400e+06,2192100.0,4177900.0,4741500.0,1.955100e+06,62652.0,204770.0,412270.0,...,95766000,105240000,62849000,6462000,17795000,25091000,81205000,83858000,46621000,F1MVK1
3,CON__ENSEMBL:ENSBTAP00000016046;tr|A0A3Q1MJB2|...,1504800.0,1.663200e+06,748200.0,808240.0,1652400.0,8.057300e+05,322920.0,527370.0,740700.0,...,9164700,9706700,7764600,11097000,12730000,11967000,12580000,11107000,15983000,A0A3Q1MJB2
4,CON__ENSEMBL:ENSBTAP00000018229;sp|Q3MHN5|VTDB...,417430.0,2.806600e+05,49046.0,61520.0,532010.0,6.678900e+04,102500.0,158930.0,1177100.0,...,0,4422300,0,0,3328100,8531800,9362800,4650600,5863800,Q3MHN5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
455,tr|Q91XL1|Q91XL1_MOUSE,108820.0,7.017700e+04,136220.0,367560.0,479590.0,1.951600e+05,0.0,0.0,0.0,...,1496800,1664800,830110,0,0,0,0,0,0,Q91XL1
456,tr|Q9CPN9|Q9CPN9_MOUSE,0.0,0.000000e+00,0.0,2639800.0,3958900.0,3.235200e+06,0.0,3514400.0,0.0,...,0,0,0,0,22453000,0,0,0,0,Q9CPN9
457,tr|Q9D6T8|Q9D6T8_MOUSE;tr|D3Z724|D3Z724_MOUSE;...,0.0,7.987600e+05,0.0,0.0,0.0,0.000000e+00,760870.0,0.0,0.0,...,0,0,0,10729000,0,0,0,0,0,Q9D6T8
458,tr|Q9JLI2|Q9JLI2_MOUSE,816390.0,7.248700e+05,469570.0,1431300.0,2710400.0,4.253200e+05,0.0,0.0,0.0,...,32043000,55790000,21094000,0,0,0,0,0,0,Q9JLI2


In [83]:
#df = clean_dfs[0].merge(clean_dfs[1], on=["Protein IDs", "uniprot"], how="outer")

In [84]:
#df.columns

In [85]:
folder_path = output_path / "02_TP_FP_annotated"
if not os.path.exists(folder_path):
    os.mkdir(folder_path)


annotated_clean_dfs = []
    
for df in clean_dfs:
    name = df.columns.tolist()[1].split("_", 1)[0]
    
    #remove keratins 
    #print(len(df))
    #df = df[~df["uniprot_id"].isin(keratins_list)]

    #annotate TP FP
    print(len(df))
    df = df.rename(columns={"uniprot_id": "Entry"})
    merged = df.merge(master_list[["annotation", "Entry"]], on="Entry", how="left")
    merged = merged.drop_duplicates()
    merged = merged.rename(columns={"Entry": "uniprot_id"})
    print(len(merged))

    merged_df = merged.set_index(["uniprot_id", "Protein IDs", "annotation"])

    merged_df.to_csv(folder_path / (name + ".csv"))
    annotated_clean_dfs.append(merged_df)

320
320
460
460


In [88]:
def get_ratio_condition_df(sub_df, cond_columns, ctrl_columns):
    for condition in cond_columns:
        for control in ctrl_columns:
            sub_df["ratio_"+condition+"/"+control] = sub_df[condition] / sub_df[control]
        
    ratio_df = sub_df.filter(regex='ratio_')
    return(ratio_df)

In [89]:
folder_path = output_path / "03_ratios"
if not os.path.exists(folder_path):
    os.mkdir(folder_path)


for df in annotated_clean_dfs:
    name = df.columns.tolist()[1].split("_", 1)[0]
    
    intensities = ["_iBAQ_", "_LFQ_"]
    
    int_list = []
    for intensity in intensities: 
        
        filter_cols = df.filter(regex=intensity).columns
        sub_df = df[filter_cols]
        
        control_labelling = "Cre-"
        treatment_labelling = "Cre+"
        
        cond_columns = sub_df.filter(like=treatment_labelling).columns.tolist()
        ctrl_columns = sub_df.filter(like=control_labelling).columns.tolist()
        ratio_df = get_ratio_condition_df(sub_df, cond_columns, ctrl_columns)
        
        int_list.append(ratio_df)
        
        #ratio_dfs_dict[condition] = ratio_df
        
        #ratio_dfs_dict = {}
    
    col_list = ['uniprot_id', 'Protein IDs', 'annotation']
    ratio_table = reduce(lambda df1,df2: pd.merge(df1,df2,on=col_list, how="outer"), int_list)
    
    #print(ratio_table.head())
    
    #merge with SI
    ratio_and_signal_intensity = ratio_table.merge(df, on=['uniprot_id', 'Protein IDs', 'annotation'], how='left')
    ratio_and_signal_intensity.to_csv(folder_path / (name + ".csv"))        
        